# 06 — Attribution Engine
### Brand Sentiment Monitor

Validates the hybrid attribution engine (`src/attribution/engine.py`) — the core
differentiator that converts sentence-level sentiment into **brand-level sentiment**.

No training in this notebook. Tests the three sub-modules:
- **5.1** Comparative pattern detector — "Nike better than Puma"
- **5.2** Default attribution — single brand → full sentiment
- **5.3** Conflict resolver — multi-sentence, same brand appears twice

**EDA decisions validated here:**

| Finding | Validation |
|---------|------------|
| 23 | Multi-sentence posts — position-aware attribution tested |
| 25 | Co-mention rate is low — default path is majority |

**Outputs:**
```
outputs/reports/attribution_eval.json    validation results
outputs/visualizations/06_attribution_routes.png
```

---
Run notebooks 00 → 05 first.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, sys, json, re, warnings
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

DRIVE_ROOT  = "/content/drive/MyDrive/brand-sentiment-monitor"
OUTPUTS_VIZ = os.path.join(DRIVE_ROOT, "outputs/visualizations")
OUTPUTS_RPT = os.path.join(DRIVE_ROOT, "outputs/reports")
KAGGLE_PROC = os.path.join(DRIVE_ROOT, "data/kaggle/processed")

for d in [OUTPUTS_VIZ, OUTPUTS_RPT]:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, DRIVE_ROOT)
sys.path.insert(0, os.path.join(DRIVE_ROOT, "src"))
print("Paths set ✅")


Mounted at /content/drive
Paths set ✅


## 1. Load Attribution Engine

In [ ]:
from attribution.engine import (
    attribute,
    resolve_conflicts,
    BrandSentiment,
    COMPARISON_PATTERNS,
    FLIP,
)
from brand.detector import detect, detect_with_positions

print(f"Attribution engine loaded ✅")
print(f"  Comparison patterns : {len(COMPARISON_PATTERNS)}")
print(f"  FLIP map            : {FLIP}")
print()
print("Comparison patterns available:")
for pat, b_sent, a_sent in COMPARISON_PATTERNS:
    print(f"  {str(pat.pattern):<35}  before={b_sent}  after={a_sent}")


Attribution engine loaded ✅
  Comparison patterns : 10
  FLIP map            : {'positive': 'negative', 'negative': 'positive', 'neutral': 'neutral'}

Comparison patterns available:
  \bbetter\s+than\b                    before=positive  after=negative
  \bworse\s+than\b                     before=negative  after=positive
  \bbeats\b                            before=positive  after=negative
  \boutperforms\b                      before=positive  after=negative
  \bsuperior\s+to\b                    before=positive  after=negative
  \bprefer\b.{0,30}\bover\b            before=positive  after=negative
  \bwins\s+over\b                      before=positive  after=negative
  \bswitched\s+from\b                  before=negative  after=positive
  \bunlike\b                           before=negative  after=positive
  \bnot\s+as\s+good\s+as\b             before=negative  after=positive


## 2. Sub-module 5.1 — Comparative Pattern Tests

Each case tests: correct brand detection + correct sentiment assignment
based on position relative to the comparison keyword.

In [ ]:
comparative_cases = [
    # (text, brand_before, expected_before_sent, brand_after, expected_after_sent)
    ("Nike is better than Puma for running",
     "Nike", "positive", "Puma", "negative"),
    ("Adidas beats Brooks on cushioning",
     "Adidas", "positive", "Brooks", "negative"),
    ("I switched from Nike to Hoka last month",
     "Nike", "negative", "Hoka", "positive"),
    ("Puma is not as good as Adidas for training",
     "Puma", "negative", "Adidas", "positive"),
    ("Hoka wins over Brooks for ultra running",
     "Hoka", "positive", "Brooks", "negative"),
    ("Saucony outperforms New Balance on the track",
     "Saucony", "positive", "NewBalance", "negative"),
]

# Stub sentiment/sarcasm (real models not needed — engine logic is pure Python)
STUB_SENT_POS = {"label": "positive", "score": 0.90}
STUB_SENT_NEG = {"label": "negative", "score": 0.85}
STUB_NO_SARC  = {"is_sarcastic": False, "score": 0.05}

passed, failed = 0, 0
print("Comparative Attribution Tests:")
print(f"  {'Pass':>4}  {'B-before':>12} {'S-before':>10}  {'B-after':>12} {'S-after':>10}  Text")
print("  " + "─" * 80)

for text, b_before, exp_b, b_after, exp_a in comparative_cases:
    brands    = detect(text)
    positions = detect_with_positions(text)
    results   = attribute(text, brands, STUB_SENT_POS, STUB_NO_SARC, positions)

    result_map = {r.brand: r.sentiment for r in results}
    ok_b = result_map.get(b_before) == exp_b
    ok_a = result_map.get(b_after)  == exp_a
    ok   = ok_b and ok_a
    status = "✅" if ok else "❌"
    if ok: passed += 1
    else:  failed += 1

    got_b = result_map.get(b_before, "NOT FOUND")
    got_a = result_map.get(b_after,  "NOT FOUND")
    print(f"  {status}  {b_before:>12} {got_b:>10}  {b_after:>12} {got_a:>10}  {text[:55]}")

print(f"\nComparative tests: {passed}/{passed+failed} passed")


Comparative Attribution Tests:
  Pass      B-before   S-before       B-after    S-after  Text
  ────────────────────────────────────────────────────────────────────────────────
  ✅          Nike   positive          Puma   negative  Nike is better than Puma for running
  ❌        Adidas   positive        Brooks  NOT FOUND  Adidas beats Brooks on cushioning
  ❌          Nike   positive          Hoka   positive  I switched from Nike to Hoka last month
  ✅          Puma   negative        Adidas   positive  Puma is not as good as Adidas for training
  ❌          Hoka   positive        Brooks  NOT FOUND  Hoka wins over Brooks for ultra running
  ✅       Saucony   positive    NewBalance   negative  Saucony outperforms New Balance on the track

Comparative tests: 3/6 passed


## 3. Sub-module 5.2 — Default Attribution Tests

In [ ]:
default_cases = [
    # (text, expected_brand, expected_sentiment, overall_sentiment_label)
    ("I love Nike shoes, best purchase ever",         "Nike",    "positive", "positive"),
    ("These Adidas shoes completely fell apart",      "Adidas",  "negative", "negative"),
    ("Nike and Adidas both released new drops",       "Nike",    "positive", "positive"),
    ("Nike and Adidas both released new drops",       "Adidas",  "positive", "positive"),
    ("Puma has decent quality but nothing special",   "Puma",    "neutral",  "neutral"),
]

passed_d, failed_d = 0, 0
print("Default Attribution Tests:")
print(f"  {'Pass':>4}  {'Brand':>10} {'Expected':>10} {'Got':>10}  Text")
print("  " + "─" * 70)

for text, exp_brand, exp_sent, overall_sent in default_cases:
    brands    = detect(text)
    positions = detect_with_positions(text)
    stub_sent = {"label": overall_sent, "score": 0.88}
    results   = attribute(text, brands, stub_sent, STUB_NO_SARC, positions)

    result_map = {r.brand: r for r in results}
    got_sent   = result_map.get(exp_brand)
    ok         = got_sent is not None and got_sent.sentiment == exp_sent
    status     = "✅" if ok else "❌"
    if ok: passed_d += 1
    else:  failed_d += 1

    got_str = got_sent.sentiment if got_sent else "NOT FOUND"
    print(f"  {status}  {exp_brand:>10} {exp_sent:>10} {got_str:>10}  {text[:55]}")

print(f"\nDefault attribution tests: {passed_d}/{passed_d+failed_d} passed")


Default Attribution Tests:
  Pass       Brand   Expected        Got  Text
  ──────────────────────────────────────────────────────────────────────
  ✅        Nike   positive   positive  I love Nike shoes, best purchase ever
  ✅      Adidas   negative   negative  These Adidas shoes completely fell apart
  ✅        Nike   positive   positive  Nike and Adidas both released new drops
  ✅      Adidas   positive   positive  Nike and Adidas both released new drops
  ✅        Puma    neutral    neutral  Puma has decent quality but nothing special

Default attribution tests: 5/5 passed


## 4. Sarcasm Flip Tests

In [ ]:
sarcasm_cases = [
    # (text, overall_sentiment_without_flip, is_sarcastic, expected_brand_sentiment)
    ("Oh great, my Nike shoes fell apart after two days. Love it.",
     "positive", True, "negative"),
    ("Absolutely love how Puma shoes break in week one. Fantastic quality.",
     "positive", True, "negative"),
    ("These Adidas shoes are genuinely amazing, best I have owned.",
     "positive", False, "positive"),
    ("Terrible Reebok quality as always, no surprise there.",
     "negative", True, "negative"),   # sarcasm on negative → stays negative (neutral flip)
]

passed_s, failed_s = 0, 0
print("Sarcasm Flip Tests:")
print(f"  {'Pass':>4}  {'Is-sarc':>8} {'Expected':>10} {'Got':>10}  Text")
print("  " + "─" * 70)

for text, overall, is_sarc, exp_sent in sarcasm_cases:
    brands    = detect(text)
    positions = detect_with_positions(text)
    stub_sent = {"label": overall, "score": 0.90}
    stub_sarc = {"is_sarcastic": is_sarc, "score": 0.85 if is_sarc else 0.10}
    results   = attribute(text, brands, stub_sent, stub_sarc, positions)

    got_sent = results[0].sentiment if results else "NO BRAND"
    ok       = got_sent == exp_sent
    status   = "✅" if ok else "❌"
    if ok: passed_s += 1
    else:  failed_s += 1

    print(f"  {status}  {str(is_sarc):>8} {exp_sent:>10} {got_sent:>10}  {text[:60]}")

print(f"\nSarcasm flip tests: {passed_s}/{passed_s+failed_s} passed")


Sarcasm Flip Tests:
  Pass   Is-sarc   Expected        Got  Text
  ──────────────────────────────────────────────────────────────────────
  ✅      True   negative   negative  Oh great, my Nike shoes fell apart after two days. Love it.
  ✅      True   negative   negative  Absolutely love how Puma shoes break in week one. Fantastic 
  ✅     False   positive   positive  These Adidas shoes are genuinely amazing, best I have owned.
  ❌      True   negative   positive  Terrible Reebok quality as always, no surprise there.

Sarcasm flip tests: 3/4 passed


## 5. Sub-module 5.3 — Conflict Resolver

In [ ]:
# Conflict resolver: brand appears in multiple sentences with different sentiments
# resolve_conflicts keeps the highest-confidence result

from attribution.engine import resolve_conflicts, BrandSentiment

conflict_inputs = [
    [
        BrandSentiment(brand="Nike", sentiment="positive", score=0.92, is_sarcastic=False, method="default"),
        BrandSentiment(brand="Nike", sentiment="negative", score=0.71, is_sarcastic=False, method="default"),
    ],
    [
        BrandSentiment(brand="Adidas", sentiment="negative", score=0.88, is_sarcastic=False, method="comparative"),
        BrandSentiment(brand="Puma",   sentiment="positive", score=0.85, is_sarcastic=False, method="comparative"),
        BrandSentiment(brand="Adidas", sentiment="positive", score=0.60, is_sarcastic=False, method="default"),
    ],
]
expected_outcomes = [
    {"Nike": "positive"},           # 0.92 wins over 0.71
    {"Adidas": "negative", "Puma": "positive"},  # 0.88 wins over 0.60
]

print("Conflict Resolver Tests:")
all_ok_c = True
for i, (inputs, expected) in enumerate(zip(conflict_inputs, expected_outcomes)):
    resolved   = resolve_conflicts(inputs)
    result_map = {r.brand: r.sentiment for r in resolved}
    ok         = result_map == expected
    all_ok_c   = all_ok_c and ok
    print(f"  Case {i+1}: {'✅' if ok else '❌'}")
    print(f"    Input  : {[(r.brand, r.sentiment, r.score) for r in inputs]}")
    print(f"    Got    : {result_map}")
    print(f"    Expected: {expected}")
    print()

print(f"Conflict resolver: {'✅ all passed' if all_ok_c else '❌ some failed'}")


Conflict Resolver Tests:
  Case 1: ✅
    Input  : [('Nike', 'positive', 0.92), ('Nike', 'negative', 0.71)]
    Got    : {'Nike': 'positive'}
    Expected: {'Nike': 'positive'}

  Case 2: ✅
    Input  : [('Adidas', 'negative', 0.88), ('Puma', 'positive', 0.85), ('Adidas', 'positive', 0.6)]
    Got    : {'Adidas': 'negative', 'Puma': 'positive'}
    Expected: {'Adidas': 'negative', 'Puma': 'positive'}

Conflict resolver: ✅ all passed


## 6. End-to-End Pipeline Routing Analysis

Run 200 random brand-relevant posts through the full attribution pipeline
and measure what fraction takes each routing path.

In [ ]:
import random
random.seed(42)

# Load some real texts from s140
s140 = pd.read_csv(os.path.join(KAGGLE_PROC, "s140_clean.csv"))
sample_texts = s140["text"].dropna().astype(str).sample(50_000, random_state=42).tolist()

# Find brand-relevant texts
brand_relevant = [t for t in sample_texts if detect(t)]
sample_200     = random.sample(brand_relevant, min(200, len(brand_relevant)))

routes = {"comparative": 0, "default": 0, "no_brand": 0}
for text in sample_200:
    brands    = detect(text)
    positions = detect_with_positions(text)
    stub_sent = {"label": "positive", "score": 0.80}
    stub_sarc = {"is_sarcastic": False, "score": 0.05}

    if not brands:
        routes["no_brand"] += 1
        continue

    results = attribute(text, brands, stub_sent, stub_sarc, positions)
    if results and results[0].method == "comparative":
        routes["comparative"] += 1
    else:
        routes["default"] += 1

total = sum(routes.values())
print(f"Attribution routing on {total} brand-relevant posts:")
for route, count in routes.items():
    pct = count / max(total, 1) * 100
    bar = "█" * int(pct / 2)
    print(f"  {route:<15} {count:>5}  ({pct:.1f}%)  {bar}")

print()
print(f"EDA Finding 25: brand co-mentions rare → default path should dominate")
comp_pct = routes['comparative'] / max(total,1) * 100
if comp_pct < 10:
    print(f"✅ Confirmed — comparative path = {comp_pct:.1f}% (< 10%)")
else:
    print(f"ℹ️  Comparative path = {comp_pct:.1f}% — higher than expected")


Attribution routing on 17 brand-relevant posts:
  comparative         0  (0.0%)  
  default            17  (100.0%)  ██████████████████████████████████████████████████
  no_brand            0  (0.0%)  

EDA Finding 25: brand co-mentions rare → default path should dominate
✅ Confirmed — comparative path = 0.0% (< 10%)


In [ ]:
# Routing pie chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Routing distribution
labels_r = [k for k, v in routes.items() if v > 0]
sizes_r  = [v for v in routes.values() if v > 0]
colors_r = ["#e74c3c", "#3498db", "#95a5a6"][:len(labels_r)]
axes[0].pie(sizes_r, labels=labels_r, autopct="%1.1f%%", colors=colors_r,
            startangle=90, textprops={"fontsize": 11})
axes[0].set_title("Attribution Routing Distribution", fontweight="bold")

# Test summary bar
test_results = {
    "Comparative\n(5.1)": (passed, failed),
    "Default\n(5.2)"    : (passed_d, failed_d),
    "Sarcasm flip"       : (passed_s, failed_s),
    "Conflict\n(5.3)"   : (sum(1 for i in range(len(conflict_inputs)) if i < len(conflict_inputs)), 0),
}
test_names   = list(test_results.keys())
test_pass    = [v[0] for v in test_results.values()]
test_fail    = [v[1] for v in test_results.values()]
x = range(len(test_names))
axes[1].bar(x, test_pass, color="#2ecc71", label="Pass", alpha=0.85)
axes[1].bar(x, test_fail, bottom=test_pass, color="#e74c3c", label="Fail", alpha=0.85)
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(test_names, fontsize=9)
axes[1].set_title("Unit Test Results per Sub-module", fontweight="bold")
axes[1].legend(); axes[1].set_ylabel("Test cases")
axes[1].grid(axis="y", alpha=0.3)

plt.suptitle("Attribution Engine — Validation Summary", fontweight="bold")
plt.tight_layout()
out_path = os.path.join(OUTPUTS_VIZ, "06_attribution_routes.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")


Saved → /content/drive/MyDrive/brand-sentiment-monitor/outputs/visualizations/06_attribution_routes.png


## 7. Save Attribution Evaluation Report

In [ ]:
total_tests  = (passed + failed) + (passed_d + failed_d) + (passed_s + failed_s) + len(conflict_inputs)
total_passed = passed + passed_d + passed_s + (len(conflict_inputs) if all_ok_c else 0)

attribution_report = {
    "engine_path" : "src/attribution/engine.py",
    "num_comparison_patterns": len(COMPARISON_PATTERNS),
    "unit_tests": {
        "comparative_5_1": {"passed": passed, "failed": failed},
        "default_5_2"    : {"passed": passed_d, "failed": failed_d},
        "sarcasm_flip"   : {"passed": passed_s, "failed": failed_s},
        "conflict_5_3"   : {"passed": len(conflict_inputs) if all_ok_c else 0,
                            "failed": 0 if all_ok_c else len(conflict_inputs)},
        "total_passed"   : total_passed,
        "total_tests"    : total_tests,
    },
    "routing_analysis": {
        "sample_size"       : total,
        "comparative_pct"   : round(routes["comparative"] / max(total,1) * 100, 2),
        "default_pct"       : round(routes["default"]     / max(total,1) * 100, 2),
        "finding_25_confirmed": comp_pct < 10,
    },
    "comparison_patterns": [str(p.pattern) for p, _, _ in COMPARISON_PATTERNS],
    "sarcasm_flip_map"    : FLIP,
    "outputs": {
        "routes_png" : "outputs/visualizations/06_attribution_routes.png",
        "eval_json"  : "outputs/reports/attribution_eval.json",
    },
}

rpt_path = os.path.join(OUTPUTS_RPT, "attribution_eval.json")
with open(rpt_path, "w") as f:
    json.dump(attribution_report, f, indent=2)

print(f"Saved → {rpt_path}")
print()
print("=" * 58)
print("ATTRIBUTION ENGINE — VALIDATION SUMMARY")
print("=" * 58)
print(f"  Unit tests passed : {total_passed} / {total_tests}")
print(f"  Comparative (5.1) : {passed}/{passed+failed}")
print(f"  Default (5.2)     : {passed_d}/{passed_d+failed_d}")
print(f"  Sarcasm flip      : {passed_s}/{passed_s+failed_s}")
print(f"  Conflict (5.3)    : {'all' if all_ok_c else 'some failed'}")
print(f"  Routing analysis  : comparative={comp_pct:.1f}%  default={routes['default']/max(total,1)*100:.1f}%")
print("=" * 58)
print(f"\n✅ Notebook 06 complete. Next: 07_emotion_model.ipynb")


Saved → /content/drive/MyDrive/brand-sentiment-monitor/outputs/reports/attribution_eval.json

ATTRIBUTION ENGINE — VALIDATION SUMMARY
  Unit tests passed : 13 / 17
  Comparative (5.1) : 3/6
  Default (5.2)     : 5/5
  Sarcasm flip      : 3/4
  Conflict (5.3)    : all
  Routing analysis  : comparative=0.0%  default=100.0%

✅ Notebook 06 complete. Next: 07_emotion_model.ipynb
